# Feature extraction — Kernel PCA (RBF-PCA)

Phiên bản phi tuyến của PCA (Schölkopf, Smola & Müller, *Nonlinear component analysis as a kernel eigenvalue problem*, Neural Computation 10(5) (1998) 1299–1319): ánh xạ ngầm mỗi mẫu sang không gian đặc trưng 𝓗 qua kernel k(x, x') = ⟨φ(x), φ(x')⟩ rồi làm PCA trong 𝓗. Kabir, Chen & Ludwig (*A performance analysis of dimensionality reduction algorithms in machine learning models for cancer prediction*, Healthcare Analytics 3 (2023) 100125) dùng **PCA với kernel RBF (RBF-PCA)** trên dữ liệu RNA-seq log2(x+1) của TCGA-PRAD (20 531 gene), giữ **250 thành phần** (cùng số chiều với PCA và autoencoder), không chuẩn hoá thêm; RBF-PCA + SVM đạt AUC 0.935, gần bằng autoencoder, vì RBF-PCA tạo ra dữ liệu gần như tách được tuyến tính.

- **Thuật toán (trên train, n mẫu):** K_ij = exp(−γ‖xᵢ − xⱼ‖²) → căn giữa trong 𝓗: K̃ = HKH, H = I − 1ₙ1ₙᵀ/n → phân rã trị riêng K̃ = AΛAᵀ, giữ `N_FEATURES` cặp (λ, α) lớn nhất → toạ độ mẫu train: **Z = A·Λ^{1/2}**. Mẫu mới x: k_x = [k(x, xᵢ)]ᵢ (căn giữa theo thống kê của train) → **z = k̃_x · A·Λ^{−1/2}**. Cài đặt bằng `sklearn.decomposition.KernelPCA(kernel="precomputed")` để ma trận kernel chỉ tính 1 lần và dùng lại cho tỉ lệ phương sai.
- **γ (bandwidth):** bài báo không nêu giá trị γ. Mặc định `KPCA_GAMMA = "median"` (median heuristic: γ = 1 / median‖xᵢ − xⱼ‖² trên train), không phụ thuộc thang đo và số gene. `KPCA_GAMMA = "sklearn"` dùng mặc định của scikit-learn γ = 1/p (p = số gene), hoặc đặt một số thực cụ thể.
- **Số thành phần:** K̃ có hạng ≤ n − 1 nên số chiều tối đa là `min(N_FEATURES, n_train − 1)`. Mặc định `N_FEATURES = 100` để so sánh cùng số chiều với các phương pháp chọn gene (bài báo: 250).
- **Khác với feature selection:** đặc trưng mới là **hàm phi tuyến của toàn bộ gene** (không còn là gene gốc). Mỗi fold: **fit chỉ trên `train`**, sau đó dùng mô hình đã fit để biến đổi `val` và `test` (val/test không tham gia fit, kernel của chúng chỉ tính với mẫu train). Dữ liệu đầu vào giữ nguyên log2 expression, không chuẩn hoá.
- **Đầu vào:** `DATA_DIR/fold_{1..5}/{train,val,test}.h5`.
- **Đổi dataset:** sửa `DATA_DIR` ở cell cấu hình. Số chiều đầu ra: `N_FEATURES` (mặc định 100).
- **Đầu ra** (`/kaggle/working/feature_extraction/kpca/<dataset>/`):
  - `features/fold_k/{train,val,test}.h5` — dữ liệu sau trích xuất, **cùng format với dữ liệu gốc** (pandas, key=`"data"`, index = sample_id, cột = `KPC1..KPCk` + `label`). Notebook phân loại chỉ cần trỏ `DATA_DIR`/`DATA_ROOT` vào thư mục `features/`.
  - `components/fold_k.csv`, `components_all_folds.csv` — thông tin từng thành phần (rank, score: tỉ lệ phương sai trong 𝓗 = λₖ / Tr(K̃))
  - `loadings_top_genes.csv` — Kernel PCA **không có** vector trọng số theo gene (thành phần nằm trong 𝓗). Thay vào đó lưu **pseudo-loading** = hệ số tương quan Pearson giữa từng gene và từng thành phần trên train; top `N_TOP_GENES` gene theo |tương quan| của mỗi thành phần (diễn giải sinh học)
  - `metrics_per_fold.csv`, `metrics_summary.csv` — thời gian, CPU utilization, peak memory, energy, EDP, carbon footprint
- **Đo lường:** `ResourceTracker` chỉ bao quanh lời gọi `fit_extractor` (tính kernel train + phân rã trị riêng + biến đổi train), không tính thời gian đọc file và pseudo-loading. Thời gian biến đổi val+test (gồm tính kernel với train) ghi riêng ở cột `transform_time_sec`. Energy/Carbon ước lượng bằng `codecarbon` (tracking_mode="process"); EDP = Energy(J) × Time(s).

In [ ]:
# tables: backend cho pd.read_hdf | codecarbon: do nang luong/CO2 | psutil: CPU & RAM
get_ipython().system("pip install -q codecarbon psutil tables")

In [ ]:
import os

# ====== CAU HINH (sua tay) ======
# Doi DATA_DIR sang bo khac (vd .../h5/tcga_luad) de chay dataset khac
DATA_DIR = "/kaggle/input/datasets/lfreedom2750/experiment-dataset/h5/gse68465"
N_FEATURES = 100          # so chieu dac trung moi sau trich xuat (bai bao: 250)
N_FOLDS = 5
SEED = 42
METHOD = "kpca"
KPCA_KERNEL = "rbf"       # bai bao: RBF | "poly" | "sigmoid" | "linear" (= PCA thuong)
KPCA_GAMMA = "median"     # "median": 1/median(||xi-xj||^2) tren train | "sklearn": 1/p | so thuc
KPCA_DEGREE = 3           # chi dung cho "poly"
KPCA_COEF0 = 1.0          # chi dung cho "poly", "sigmoid"
KPCA_EIGEN_SOLVER = "auto"  # "auto" | "dense" | "arpack" | "randomized"
METHOD_PARAMS = {"kernel": KPCA_KERNEL, "gamma": KPCA_GAMMA, "degree": KPCA_DEGREE, "coef0": KPCA_COEF0,
                 "eigen_solver": KPCA_EIGEN_SOLVER,
                 "reference": "Kabir et al., Healthcare Analytics 3 (2023) 100125; Scholkopf et al. 1998"}
# None -> codecarbon tu dinh vi (can Internet); hoac dat ma ISO 3 ky tu, vd "USA", "VNM"
COUNTRY_ISO_CODE = None

DATASET_NAME = os.path.basename(os.path.normpath(DATA_DIR))
RESULT_DIR = os.path.join("/kaggle/working/feature_extraction", METHOD, DATASET_NAME)
os.makedirs(os.path.join(RESULT_DIR, "components"), exist_ok=True)

for _s in ["train", "val", "test"]:
    assert os.path.isfile(os.path.join(DATA_DIR, "fold_1", f"{_s}.h5")), \
        f"Khong thay {DATA_DIR}/fold_1/{_s}.h5 -- da Add Input dataset chua?"
print("Dataset :", DATASET_NAME, "->", DATA_DIR)
print("Method  :", METHOD, "| N_FEATURES =", N_FEATURES)
print("Ket qua :", RESULT_DIR)
print("So CPU  :", os.cpu_count())

## Đọc dữ liệu

In [ ]:
import json, time, threading, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

STAGE_NAMES = {0: "Normal", 1: "I", 2: "II", 3: "III", 4: "IV"}


def load_split(fold, split="train"):
    # File h5 (pandas, key="data"): index = sample_id, cot = ten gene + cot "label"
    df = pd.read_hdf(os.path.join(DATA_DIR, f"fold_{fold}", f"{split}.h5"), key="data")
    X = df.drop(columns="label")
    # Giu nguyen gia tri log2 expression, KHONG chuan hoa
    return X.values.astype(np.float64), df["label"].values.astype(int), X.columns.to_numpy(), df.index.to_numpy()


# Doc thu fold 1 de xem cau truc
X0, y0, genes0, ids0 = load_split(1)
print(f"fold_1/train: {X0.shape[0]} mau x {X0.shape[1]} gene | khoang gia tri [{X0.min():.2f}, {X0.max():.2f}]")
print("Vi du gene:", list(genes0[:5]), "| vi du sample:", list(ids0[:3]))
print("Phan bo lop:", pd.Series(y0).map(STAGE_NAMES).value_counts().to_dict())
del X0, y0, genes0, ids0

## Bộ đo tài nguyên

In [ ]:
import psutil

try:
    from codecarbon import EmissionsTracker, OfflineEmissionsTracker
    _CODECARBON = True
except ImportError:
    _CODECARBON = False


class ResourceTracker:
    """Do tren 1 khoi lenh: thoi gian (s), CPU utilization, peak memory (RSS, gom ca
    tien trinh con cua joblib), energy (kWh, codecarbon), carbon (kg CO2eq) va
    Energy-Delay Product EDP = Energy(J) * Time(s)."""

    def __init__(self, name, interval=0.05):
        self.name, self.interval = name, interval
        self.proc = psutil.Process()
        self._stop = threading.Event()

    def _procs(self):
        try:
            return [self.proc] + self.proc.children(recursive=True)
        except psutil.Error:
            return [self.proc]

    def _poll(self):
        # RSS tong va CPU time cua worker con (tru phan CPU worker da dung truoc khi bat dau do)
        rss = 0
        for p in self._procs():
            try:
                rss += p.memory_info().rss
                if p.pid != self.proc.pid:
                    t = p.cpu_times()
                    self._child_cpu[p.pid] = t.user + t.system - self._child_base.get(p.pid, 0.0)
            except psutil.Error:
                pass
        self._peak_rss = max(self._peak_rss, rss)
        return rss

    def _sampler(self):
        while not self._stop.is_set():
            self._poll()
            self._sys_cpu.append(psutil.cpu_percent(interval=None))
            self._stop.wait(self.interval)

    def __enter__(self):
        self._child_cpu, self._sys_cpu, self._peak_rss = {}, [], 0
        self._child_base = {}
        for p in self._procs()[1:]:
            try:
                t = p.cpu_times()
                self._child_base[p.pid] = t.user + t.system
            except psutil.Error:
                pass
        self.tracker = None
        if _CODECARBON:
            try:
                kw = dict(project_name=self.name, output_dir=RESULT_DIR, save_to_file=False,
                          log_level="error", measure_power_secs=1, tracking_mode="process")
                self.tracker = (OfflineEmissionsTracker(country_iso_code=COUNTRY_ISO_CODE, **kw)
                                if COUNTRY_ISO_CODE else EmissionsTracker(**kw))
                self.tracker.start()
            except Exception as e:
                print("[ResourceTracker] codecarbon loi:", e)
                self.tracker = None
        self.base_rss = self._poll()
        self._child_cpu = {}
        psutil.cpu_percent(interval=None)
        t = self.proc.cpu_times()
        self._cpu0 = t.user + t.system
        self._stop.clear()
        self._thread = threading.Thread(target=self._sampler, daemon=True)
        self._thread.start()
        self._t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        self.time_sec = time.perf_counter() - self._t0
        t = self.proc.cpu_times()
        main_cpu = t.user + t.system - self._cpu0
        self._stop.set()
        self._thread.join(timeout=2)
        self._poll()
        cpu_sec = main_cpu + sum(self._child_cpu.values())

        self.energy_kwh, self.carbon_kg = float("nan"), float("nan")
        if self.tracker is not None:
            try:
                self.carbon_kg = float(self.tracker.stop() or 0.0)
                data = self.tracker.final_emissions_data
                self.energy_kwh = float(data.energy_consumed) if data is not None else float("nan")
            except Exception as e:
                print("[ResourceTracker] codecarbon stop loi:", e)

        n_cpu = os.cpu_count() or 1
        self.metrics = {
            "time_sec": self.time_sec,
            "cpu_time_sec": cpu_sec,
            # % tren tong nang luc may (100% = dung het n_cpu core)
            "cpu_util_percent": 100.0 * cpu_sec / (self.time_sec * n_cpu) if self.time_sec > 0 else float("nan"),
            "avg_cores_used": cpu_sec / self.time_sec if self.time_sec > 0 else float("nan"),
            "system_cpu_percent_mean": float(np.mean(self._sys_cpu)) if self._sys_cpu else float("nan"),
            "peak_memory_mb": self._peak_rss / 1024 ** 2,
            "peak_memory_increase_mb": (self._peak_rss - self.base_rss) / 1024 ** 2,
            "energy_kwh": self.energy_kwh,
            "energy_joule": self.energy_kwh * 3.6e6,
            "edp_joule_sec": self.energy_kwh * 3.6e6 * self.time_sec,
            "carbon_kg_co2eq": self.carbon_kg,
            "carbon_g_co2eq": self.carbon_kg * 1e3,
        }
        return False

## Thuật toán: Kernel PCA (RBF-PCA, Kabir et al. 2023)

In [ ]:
PREFIX = "KPC"       # ten cot dac trung moi: KPC1, KPC2, ...
N_TOP_GENES = 20     # so gene top |loading| luu cho moi thanh phan
SELECT_INFO = {}   # thong tin phu cua lan fit gan nhat, duoc ghi vao metrics_per_fold.csv

def top_loading_genes(W, genes, comp_names, n_top):
    # W: (n_comp, n_gene) -> top n_top gene co |trong so| lon nhat cua moi thanh phan
    rows = []
    for c, w in zip(comp_names, np.asarray(W)):
        idx = np.argsort(-np.abs(w), kind="stable")[:n_top]
        rows += [{"component": c, "rank": r + 1, "gene": genes[i], "loading": float(w[i])}
                 for r, i in enumerate(idx)]
    return pd.DataFrame(rows)

from sklearn.decomposition import KernelPCA
from sklearn.metrics.pairwise import euclidean_distances, pairwise_kernels


def resolve_k(X, y):
    # Kernel da can giua co hang <= n - 1
    return min(N_FEATURES, X.shape[0] - 1, X.shape[1])


def resolve_gamma(X):
    if KPCA_GAMMA == "median":
        d2 = euclidean_distances(X, squared=True)
        return 1.0 / float(np.median(d2[np.triu_indices_from(d2, k=1)]))
    if KPCA_GAMMA == "sklearn":
        return 1.0 / X.shape[1]
    return float(KPCA_GAMMA)


class KPCAModel:
    """KernelPCA tren kernel tinh san: giu X_train de tinh kernel cho mau moi."""

    def __init__(self, kpca, X_train, Z_train, kernel_params):
        self.kpca, self.X_train, self.Z_train, self.kernel_params = kpca, X_train, Z_train, kernel_params

    def kernel(self, X):
        return pairwise_kernels(X, self.X_train, metric=KPCA_KERNEL, filter_params=True, **self.kernel_params)


def fit_extractor(X, y, k):
    gamma = resolve_gamma(X) if KPCA_KERNEL != "linear" else None
    params = {"gamma": gamma, "degree": KPCA_DEGREE, "coef0": KPCA_COEF0}
    K = pairwise_kernels(X, metric=KPCA_KERNEL, filter_params=True, **params)
    kpca = KernelPCA(n_components=k, kernel="precomputed", eigen_solver=KPCA_EIGEN_SOLVER, random_state=SEED)
    Z = kpca.fit_transform(K)
    # Ti le phuong sai trong H: lambda_k / Tr(K_centered), voi Tr(HKH) = Tr(K) - sum(K)/n
    total = float(np.trace(K) - K.sum() / K.shape[0])
    evr = kpca.eigenvalues_ / total
    SELECT_INFO.update(gamma=gamma, kernel=KPCA_KERNEL,
                       explained_variance_cum=float(evr.sum()),
                       n_pc_90pct_var=int(np.searchsorted(np.cumsum(evr), 0.90) + 1) if evr.sum() >= 0.90 else None)
    return KPCAModel(kpca, X, Z, params), Z, evr


def transform(model, X):
    return model.kpca.transform(model.kernel(X))


def loadings(model):
    # Pseudo-loading: tuong quan Pearson gene x thanh phan tren train (gene co phuong sai 0 -> 0)
    Xc = model.X_train - model.X_train.mean(0)
    Zc = model.Z_train - model.Z_train.mean(0)
    sx, sz = np.sqrt((Xc ** 2).sum(0)), np.sqrt((Zc ** 2).sum(0))
    sx[sx == 0], sz[sz == 0] = np.inf, np.inf
    return (Zc / sz).T @ (Xc / sx)

## Chạy trên 5 fold

In [ ]:
all_rows, metric_rows, loading_rows = [], [], []
VIS = {}   # luu fold 1 de ve hinh

for fold in range(1, N_FOLDS + 1):
    X, y, genes, ids = load_split(fold, "train")   # CHI dung tap train de fit
    Xv, yv, genes_v, ids_v = load_split(fold, "val")
    Xt, yt, genes_t, ids_t = load_split(fold, "test")
    assert (genes_v == genes).all() and (genes_t == genes).all(), "Thu tu gene train/val/test khac nhau"
    k = resolve_k(X, y)

    with ResourceTracker(f"{METHOD}_{DATASET_NAME}_fold{fold}") as rt:
        model, Z, scores = fit_extractor(X, y, k)

    t0 = time.perf_counter()
    Zv, Zt = transform(model, Xv), transform(model, Xt)
    transform_time = time.perf_counter() - t0

    names = [f"{PREFIX}{i}" for i in range(1, Z.shape[1] + 1)]
    out_dir = os.path.join(RESULT_DIR, "features", f"fold_{fold}")
    os.makedirs(out_dir, exist_ok=True)
    for split, Zs, ys, idx_s in [("train", Z, y, ids), ("val", Zv, yv, ids_v), ("test", Zt, yt, ids_t)]:
        df = pd.DataFrame(np.asarray(Zs, dtype=np.float64), index=idx_s, columns=names)
        df["label"] = ys
        df.to_hdf(os.path.join(out_dir, f"{split}.h5"), key="data", mode="w")

    comp = pd.DataFrame({"fold": fold, "rank": np.arange(1, len(names) + 1), "component": names, "score": scores})
    comp.to_csv(os.path.join(RESULT_DIR, "components", f"fold_{fold}.csv"), index=False)
    all_rows.append(comp)
    W = loadings(model)
    if W is not None:
        loading_rows.append(top_loading_genes(W, genes, names, N_TOP_GENES).assign(fold=fold))

    m = {"dataset": DATASET_NAME, "method": METHOD, "fold": fold,
         "n_train": X.shape[0], "n_val": Xv.shape[0], "n_test": Xt.shape[0],
         "n_genes_total": X.shape[1], "n_components": Z.shape[1],
         "transform_time_sec": transform_time, **SELECT_INFO, **rt.metrics}
    metric_rows.append(m)
    print(f"fold {fold}: {X.shape} -> {Z.shape[1]} chieu | {m['time_sec']:.2f}s | CPU {m['cpu_util_percent']:.1f}% "
          f"({m['avg_cores_used']:.2f} core) | peak {m['peak_memory_mb']:.0f} MB | "
          f"{m['energy_kwh']:.3e} kWh | {m['carbon_g_co2eq']:.4f} gCO2eq | transform {transform_time:.2f}s")
    if fold == 1:
        VIS = {"train": (np.asarray(Z), y), "test": (np.asarray(Zt), yt), "names": names}
    del X, Xv, Xt, model

components = pd.concat(all_rows, ignore_index=True)
components.to_csv(os.path.join(RESULT_DIR, "components_all_folds.csv"), index=False)
if loading_rows:
    pd.concat(loading_rows, ignore_index=True).to_csv(os.path.join(RESULT_DIR, "loadings_top_genes.csv"), index=False)

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(os.path.join(RESULT_DIR, "metrics_per_fold.csv"), index=False)
metrics

## Tổng hợp độ đo

In [ ]:
cols = ["time_sec", "cpu_time_sec", "cpu_util_percent", "avg_cores_used", "system_cpu_percent_mean",
        "peak_memory_mb", "peak_memory_increase_mb", "energy_kwh", "energy_joule",
        "edp_joule_sec", "carbon_kg_co2eq", "carbon_g_co2eq", "transform_time_sec"]
summary = metrics[cols].agg(["mean", "std", "min", "max"]).T
summary.loc["time_sec", "total"] = metrics["time_sec"].sum()
summary.loc["energy_kwh", "total"] = metrics["energy_kwh"].sum()
summary.loc["carbon_kg_co2eq", "total"] = metrics["carbon_kg_co2eq"].sum()
summary.to_csv(os.path.join(RESULT_DIR, "metrics_summary.csv"))

with open(os.path.join(RESULT_DIR, "config.json"), "w") as f:
    json.dump({"dataset": DATASET_NAME, "data_dir": DATA_DIR, "method": METHOD, "n_features": N_FEATURES,
               "n_folds": N_FOLDS, "seed": SEED, "params": METHOD_PARAMS, "cpu_count": os.cpu_count(),
               "country_iso_code": COUNTRY_ISO_CODE}, f, indent=1, default=str)
summary

## Trực quan hoá không gian mới (tham khảo)

In [ ]:
# Tham khao nhanh: 2 chieu dau tien cua fold 1 (train: cham tron, test: dau x), to mau theo stage
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.5, 5))
names = VIS["names"]
if len(names) >= 2:
    for split, marker, alpha in [("train", "o", 0.45), ("test", "x", 0.9)]:
        Zs, ys = VIS[split]
        for c in np.unique(ys):
            mk = ys == c
            ax.scatter(Zs[mk, 0], Zs[mk, 1], marker=marker, s=18, alpha=alpha,
                       color=plt.cm.tab10(int(c)), label=f"{STAGE_NAMES.get(int(c), c)} ({split})")
    ax.set_xlabel(names[0]); ax.set_ylabel(names[1])
else:   # chi co 1 chieu (vd LDA 2 lop) -> ve phan bo 1 chieu
    for split, marker in [("train", "o"), ("test", "x")]:
        Zs, ys = VIS[split]
        ax.scatter(Zs[:, 0], ys + (0.15 if split == "test" else 0), marker=marker, c=ys, cmap="tab10", s=18)
    ax.set_xlabel(names[0]); ax.set_ylabel("label")
ax.set_title(f"{METHOD} | {DATASET_NAME} | fold 1")
ax.legend(fontsize=7, ncol=2, loc="best")
fig.tight_layout()
fig.savefig(os.path.join(RESULT_DIR, "scatter_fold1.png"), dpi=150)
plt.show()

In [ ]:
import shutil
zip_path = shutil.make_archive(f"/kaggle/working/fe_{METHOD}_{DATASET_NAME}", "zip", RESULT_DIR)
print("Da nen:", zip_path)
for root, _, files in os.walk(RESULT_DIR):
    for fn in sorted(files):
        print(" ", os.path.relpath(os.path.join(root, fn), RESULT_DIR))